### Embeddings

In [ ]:
import spacy
# https://spacy.io/

In [2]:
#! python -m spacy download en_core_web_md
nlp = spacy.load("en_core_web_md")

In [3]:
tokens = nlp("dog cat banana")

for token in tokens:
    print(f"Palavra: {token.text}")
    print(f"Primeiros elementos vetor: {token.vector[:5]}")
    print(f"Tamanho do vetor: {len(token.vector)}\n")

Palavra: dog
Primeiros elementos vetor: [-0.72483   0.42538   0.025489 -0.39807   0.037463]
Tamanho do vetor: 300

Palavra: cat
Primeiros elementos vetor: [-0.72483   0.42538   0.025489 -0.39807   0.037463]
Tamanho do vetor: 300

Palavra: banana
Primeiros elementos vetor: [-0.6334   0.18981 -0.53544 -0.52658 -0.30001]
Tamanho do vetor: 300



In [4]:
palavra_alvo = nlp("coffe")

palavras_teste = nlp("tea drink sugar hammer")

for p in palavras_teste:
    print(f"Similaridade entre {palavra_alvo.text} e {p.text}: {palavra_alvo.similarity(p):.4f}")

Similaridade entre coffe e tea: 0.5708
Similaridade entre coffe e drink: 0.3550
Similaridade entre coffe e sugar: 0.2695
Similaridade entre coffe e hammer: 0.0104


### Chromadb

- spaCy: É o tradutor que lê o livro e resume o "sentido" dele em uma coordenada de GPS (vetor).
- ChromaDB: É o bibliotecário que guarda o livro nessa coordenada exata. Quando você faz uma pergunta, ele não procura por "palavras-chave", ele olha no mapa e vê quais livros estão estacionados perto da sua dúvida.

In [5]:
import chromadb

In [6]:
client  = chromadb.Client()
colecao = client.create_collection(name="meu_conhecimento")

In [7]:
colecao.add(
    documents=[
        "The quick brown fox jumps over the lazy dog.",
        "Artificial Intelligence is transforming the world.",
        "Python is a versatile programming language for data science.",
        "The recipe requires two cups of flour and three eggs.",
        "The cake should bake for 30 minutes at 180 degrees."
        ],
    ids=["id1", "id2", "id3", "id4", "id5"],
    metadatas=[
        {"topic": "nature", "source": "book_a"},
        {"topic": "tech", "source": "news_site"},
        {"topic": "tech", "source": "github"},
        {"topic": "cooking", "source": "grandma_recipes"},
        {"topic": "cooking", "source": "grandma_recipes"}
    ]
)

In [8]:
resultado = colecao.query(
    query_texts=["What is Python?"],
    n_results=1
)

print(f"O documento mais próximo é: {resultado['documents'][0]}")

O documento mais próximo é: ['Python is a versatile programming language for data science.']


In [9]:
resultado

{'ids': [['id3']],
 'embeddings': None,
 'documents': [['Python is a versatile programming language for data science.']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'topic': 'tech', 'source': 'github'}]],
 'distances': [[0.5105705261230469]]}

In [10]:
resultado2 = colecao.query(
    query_texts=["The impact of Artificial Intelligence"],
    n_results=2
)

In [11]:
resultado2

{'ids': [['id2', 'id3']],
 'embeddings': None,
 'documents': [['Artificial Intelligence is transforming the world.',
   'Python is a versatile programming language for data science.']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'topic': 'tech', 'source': 'news_site'},
   {'topic': 'tech', 'source': 'github'}]],
 'distances': [[0.6177182197570801, 1.584524154663086]]}

In [12]:
resultado3 = colecao.query(
    query_texts=["recipe"],
    n_results=2,
    where={"topic":"cooking"}
)

resultado3

{'ids': [['id4', 'id5']],
 'embeddings': None,
 'documents': [['The recipe requires two cups of flour and three eggs.',
   'The cake should bake for 30 minutes at 180 degrees.']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'source': 'grandma_recipes', 'topic': 'cooking'},
   {'topic': 'cooking', 'source': 'grandma_recipes'}]],
 'distances': [[0.8217378854751587, 1.316723108291626]]}